# This file builds the 2D 2-particle correlation in both the standard and WTA frames. Clusters the jets according to the two definitions, transforms things to the jet frame, makes the signal distribution, the background distribution, and then the final normalised yield.

In [1]:
import ROOT
import numpy as np
import math
import pandas as pd
import fastjet
import matplotlib.pyplot as plt
import os
import ctypes
#import uproot

In [2]:
#from preliminary file

#f = ROOT.TFile.Open("/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/data/parton_data.root")
f = ROOT.TFile.Open("/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/data/pp_parton_cascade_batch0_0.root")

tree = f.Get("trackTree") #opening up the tracktree

jet_radius = 0.8 #max radius accepted by fastjet is 1000. radius value in prl paper is 0.8 in lab frame.

wta_def = fastjet.JetDefinition(fastjet.antikt_algorithm, jet_radius, fastjet.WTA_pt_scheme) #winner take all definition
std_def = fastjet.JetDefinition(fastjet.antikt_algorithm, jet_radius) #standard E scheme definition

In [3]:
#modified from Xiao's code
#clarify what the criteria is - he had something about 200 that i have not included


def eventPass(jetPt, jetEta):  
    '''
    criteria for jet selection
    
    inputs:
        - jetEta: the jet's pseudorapidity, a positive or negative float
        - jetPt: the jet's momentum in GeV, a positive float             !!! USING STANDARD PT FOR CRITERIA, maybe later can try out with wta pt selection

    output: true if passed, false otherwise
    '''
    jetEtaCut = 1.6 #from PRL paper page 2
    jetPtCut = 550 #from prl paper,         STANDARD pt, not WTA

    if(abs(jetEta)>jetEtaCut): return False
    if(jetPt < jetPtCut): return False

    return True


def particlePass(particlePt, particleEta, particleCharge):
    '''
    criteria for particle selection

    inputs: 
        - particlePt: positive float for particle momentum magnitude
        - particleEta: float (positive or negative) for particle pseudorapidity
        - particleCharge: the particle's charge, not sure about the variable type

    output: true if passed, false otherwise
    '''
    particlePtCut = 0.3 #from prl paper page 2
    particleEtaCut = 2.4 #from prl paper page 2
    
    if(particlePt < particlePtCut): return False
    if(abs(particleEta) > particleEtaCut): return False

    if (particleCharge is None or particleCharge==0): return False #only accepting charged particles

    return True



In [4]:
#from prelim file

def make_pseudojets(pts, etas, phis, charges):
    '''
    converts particles of one jet to a list of pseudojets. particle quality cuts are built in.
    inputs:
        - pts: a vector of particles for a given jet
        - etas: the corresponding pseudorapidities for each particle in the jet
        - phis: the corresponding azimuthal angles for each particle in the jet
        - charges: the charges of each particle in the jet
        
    output:
        - pj_list: a list of pseudojets, each pseudojet represents a particle in the jet
    '''
    
    pj_list = []

    for i in range(len(pts)): #looping through each particle in the jet
        
        if particlePass(pts[i], etas[i], charges[i]): #particle selection criteria
            p = fastjet.PseudoJet() #initialising a pseudojet
            p.reset_PtYPhiM(pts[i], etas[i], phis[i], 0.0) #defining the i-th particle as a pseudojet - is mass=0 fine?
            pj_list.append(p) #adding the particle to the list of pseudojets
    
    return pj_list #list of pseudojets


In [5]:
#transforming values to jet frame


def particleToJetFrame(jet_px, jet_py, jet_pz, particle):
    """
    Transforms one constituent particle to the jet frame. eta_star quality cuts reflected in 'include' boolean.
    
    Inputs:
        - jet_px, jet_py, jet_pz: The lab frame cartesian momentum 3-vector for the WHOLE jet
        - particle: A lab frame pseudoJet CONSTITUENT of that jet

    output:
        - jt: jet frame transverse momentum of the particle
        - eta_star: jet frame pseudorapidity of the particle
        - phi_star: jet frame phi of the particle
        - include: true if particle meets eta_star and jt criteria, false otherwise
    """

    # 1. Convert to ROOT TVector3
    # Use px, py, pz to ensure we have the full 3D vector
    p_jet = ROOT.TVector3(jet_px, jet_py, jet_pz)                 #momentum 3-vector of the whole jet
    p_part = ROOT.TVector3(particle.px(), particle.py(), particle.pz()) #momentum 3-vector of the particle

    # 1. Calculate Eta star (Relative Eta), discard non-qualifiers
    theta = p_part.Angle(p_jet)     #dot product angle between particle and jet momenta, range 0 to pi
    #cases for theta
    if theta == 0 or theta == math.pi: 
        return 0, 0, 0, False #on axis, not valid eta_star
    else: 
        eta_star = -math.log(math.tan(theta / 2.0)) #pseudorapidity formula for acceptable theta
        if abs(eta_star) > 5:
            return 0, 0, 0, False #include is false because absolute eta_star greater than 5 not allowed. 

        #what about eta_star less than 0.86? that is the anti-kt jet bdry. do i need to exclude stuff outside that?

    # 2. Calculate jT (Relative pT)
    jt = p_part.Perp(p_jet) #magnitude of part of particle momentum that is perpendicular to jet momentum
    if jt < 0.3 or jt > 3: #keeping 0.3geV<jt<3GeV . Maybe not a necessary cut?
        return 0, 0, 0, False #not soft so not plotted
    
    # 4. Calculate Phi star (Relative Phi)
    unit_jet = p_jet.Unit()
    z_axis = ROOT.TVector3(0, 0, 1)
    
    # Vector purely transverse to the jet axis
    v_pt = p_part - (unit_jet * p_part.Dot(unit_jet)) #magnitude of this is jt
    
    # Define the reference plane (jet axis & beam line)
    phi_origin = unit_jet.Cross(unit_jet.Cross(z_axis)) #phi = 0 vector, taken from Xiao's code
    phi_star = v_pt.Angle(phi_origin)
    
    # Determine the sign of phi_star
    if (phi_origin.Cross(v_pt.Unit())).Dot(unit_jet) < 0:
        phi_star = -phi_star

    return jt, eta_star, phi_star, True #if we reached this point without returning, then include should be true


def jetToJetFrame(labFrame_pj):
    '''
    transforms all constituents of a full jet from lab frame to the jet frame

    input: lab_pj, a jet pseudojet with coordinates measured in lab frame. The jet should already be clustered according to the desired scheme.

    outputs: 
        - ple_pj_list, a list of valid particle pseudojets with coordinates measured in the jet frame. (excludes hard jt)
        - jet_mult: the number of particles that meet in and out of jet criteria (doesn't exclude hard jt)
        note: len(ple_pj_list) != jet_mult because jet_mult includes hard particles and failed eta_star values
    '''
    #Extracting px, py, pz of the whole jet in lab frame
    jet_px =  labFrame_pj.px()
    jet_py = labFrame_pj.py()
    jet_pz = labFrame_pj.pz()
    
    ple_pj_list = [] #initialising list of particle pseudojets

    #looping through constituents
    jet_constituents = labFrame_pj.constituents()
    jet_mult = 0 #initialising multiplicity count
    for particle in jet_constituents:
        jet_mult+=1 # a valid clustered particle, but jt and eta_star not necessarily valid

        #finding jet frame coordinates for the particle
        jetFrame_pt, jetFrame_eta, jetFrame_phi, include = particleToJetFrame(jet_px, jet_py, jet_pz, particle)

        if include == True: #particle meets eta_star and jt criteria
            p_in_jet = fastjet.PseudoJet() #initialising a pseudojet     
            p_in_jet.reset_PtYPhiM(jetFrame_pt, jetFrame_eta, jetFrame_phi, 0.0) #redefining the particle in the jet frame (last index is mass)
            ple_pj_list.append(p_in_jet) #add the newly defined particle pseudojet to the list

    return ple_pj_list, jet_mult


In [6]:
def mult_bin_check(jet_mult, mult_bin):
    '''
    checks if the jet falls within the given multiplicity bin
    input:
        - jet_mult: jet multiplicity, an integer
        - mult_bin: a list of 1 or 2 integers. 1st is lower bound, and 2nd (if given) is upper bound.
    output: True if between upper and lower bounds. If no upper bound is given, true if jet_mult is above lower bound.
    '''
    if jet_mult>=mult_bin[0]: #checking lower bound
        if len(mult_bin) == 2:  #checking if upper bound given
            return jet_mult<mult_bin[1] #true for between upper and lower bound
        else:
            return True #true for above lower bound, only one index
    return False #if below lower bound

In [7]:
#6 way histogram fill
def sixWayFill(hist, eta, phi, weight):
    '''
    helper function to do the 6-way symmetric histogram filling
    input: the 2d root histogram, the 2 inputs eta and phi, and a weight
    output: none, we are modifying the histogram
    '''
    hist.Fill(eta, phi, weight) #quadrant 1
    hist.Fill(-eta, phi, weight) #quadrant 2
    hist.Fill(-eta, -phi, weight) #quadrant 3
    hist.Fill(eta, -phi, weight) #quadrant 4
    hist.Fill(eta, 2*math.pi - phi, weight) #quadrant 1, phi wrap around pi
    hist.Fill(-eta, 2*math.pi - phi, weight) #quadrant 2, phi wrap around pi

In [8]:
# do the 2 particle correlation
#simplified version rn for testing
def buildSignal(pj_list, hSig, hEPD):
    '''
    fills in the signal and EPD histograms based on a jet represented by pj_list
    
    input: 
        - pj_list, a list of particle pseudojets for one jet. these should be in the jet frame, and in the right multiplicity bin
        - hSig, the signal histogram
        - hEPD, the event particle distribution histogram
    
    output:
        nothing. This just fills in the histograms.
    '''
    
    N_trig = len(pj_list) #JUST FOR THE LOOP, NOT THE HARD JT,FAILED ETA* INCLUSIVE MULTIPLICITY. that is jet_mult
    if N_trig < 2:
        return #no pairs can be made

    #build EPD
    for p in pj_list:
        hEPD.Fill(p.eta(), p.phi(), 1/N_trig) #filling in the trigger particle position in the EPD 

    #making the signal
    for i in range(N_trig - 1):
        trig_eta = pj_list[i].eta()
        trig_phi = pj_list[i].phi()

        for j in range(i+1, N_trig): 
            track_eta = pj_list[j].eta()
            track_phi = pj_list[j].phi()
            delta_eta_star = abs(trig_eta - track_eta) #absolute dEta
            delta_phi_star = math.acos(math.cos(trig_phi - track_phi)) #limits dPhi to [0,pi]
            #dR = math.sqrt(delta_eta_star**2 + delta_phi_star**2)

            #weight = 1/Ntrig here, but it would involve the normalised energy product for an EEC
            #filling the signal symmetrically 6 ways
            sixWayFill(hSig, delta_eta_star, delta_phi_star, 1/N_trig) #weighting by 1/number of trigger particles in this jet
    
    #contribution of this one jet to signal and EPD are now filled
    #particles and pairs are weighted per trigger in the jet


In [9]:
#build the background distribution once signal and EPD are filled
#method used here is the one described in the prl paper, not the one from github
def buildBkg(hEPD, hBkg_pre_corr, num_pairs):
    '''
    fills in the background histogram using the given EPD
    inputs:
        - hEPD, the signle-particle distribution
        - hBkg_pre_corr, the background histogram that needs filling, before it has been normalised by B(0,0)
        - num_pairs, the integer number of signal pairs.
    output:
        - fills and scales the background histogram so that it becomes hBkg, the corrected background histogram
    '''

    # 1. initialising c type coordinates
    eta1 = ctypes.c_double(0.0)
    phi1 = ctypes.c_double(0.0)
    eta2 = ctypes.c_double(0.0)
    phi2 = ctypes.c_double(0.0)

    # 2. loop to build background
    for _ in range(10*num_pairs): #background pairs are 10x the number of signal pairs, make sure it's an integer and not a float
        # Draw coords of two random particles from the EPD
        hEPD.GetRandom2(eta1, phi1)
        hEPD.GetRandom2(eta2, phi2)
        
        # 3. Extract the Python floats using .value
        e1 = eta1.value
        p1 = phi1.value
        e2 = eta2.value
        p2 = phi2.value
        
        # 4. Calculate kinematics using the extracted values
        delta_eta_star = abs(e1 - e2)
        delta_phi_star = math.acos(math.cos(p1 - p2))
        
        # 6-way fill for hBkg_pre_corr
        sixWayFill(hBkg_pre_corr, delta_eta_star, delta_phi_star, 1.0)

    #background is now filled the way described in the prl paper for the entire multiplicity bin
    #not yet corrected by B(0,0 though)

    #Also loop length is still something i've assumed...

In [10]:
#initialising root histograms

mult_bins_1 = [ [0,20], [20,30], [30,40], [40,50], [50,60], [60,69], [69,79], [79,90], [90,97], [97]] #from Xiao's code
high_mult_bin = [80] #high multiplicity jets

# eta and phi bins
eta_bins, eta_min, eta_max = 41, -6.15, 6.15 #from xiao's code. prl paper uses -3 to 3
phi_bins = 33
phi_min = -(math.pi/2.0) + (math.pi/32.0)
phi_max = (3*math.pi/2.0) + (math.pi/32.0)
phi_bin_width = (phi_max - phi_min)/phi_bins

#WTA signal and background histograms
hSig_wta = ROOT.TH2D("hSig_WTA", ";#Delta#eta*;#Delta#phi*", eta_bins, eta_min, eta_max, phi_bins, phi_min, phi_max)
hBkg_wta = ROOT.TH2D("hBkg_WTA", ";#Delta#eta*;#Delta#phi*", eta_bins, eta_min, eta_max, phi_bins, phi_min, phi_max)

#wta event particle density histogram (i.e. eta_star and phi_star of every valid trigger particle) - this is used to make the background
hEPD_wta = ROOT.TH2D("hEPD_WTA", "Background Density Map;#eta*;#phi*", 150, 0, 10, 120, -4, 4) 

#std signal, background, and EPD histograms
hSig_std = ROOT.TH2D("hSig_std", ";#Delta#eta*;#Delta#phi*", eta_bins, eta_min, eta_max, phi_bins, phi_min, phi_max)
hBkg_std = ROOT.TH2D("hBkg_std", ";#Delta#eta*;#Delta#phi*", eta_bins, eta_min, eta_max, phi_bins, phi_min, phi_max)
hEPD_std = ROOT.TH2D("hEPD_std", "Background Density Map;#eta*;#phi*", 150, 0, 10, 120, -4, 4)

In [57]:
# Helper to delete histogram objects (in case there is a memory leak warning)
#ONLY RUN IF MAKING NEW HISTOGRAMS
def clean_hist(name):
    obj = ROOT.gDirectory.Get(name)
    if obj:
        obj.Delete()

clean_hist("hSig_WTA")
clean_hist("hBkg_WTA")
clean_hist("hEPD_WTA")
clean_hist("hSig_std")
clean_hist("hBkg_std")
clean_hist("hEPD_std")

In [59]:
#to reset but not delete the histograms (i.e. wipe the data but keep the eta and phi binning)
hSig_wta.Reset()
hBkg_wta.Reset()
hEPD_wta.Reset()
hSig_std.Reset()
hBkg_std.Reset()
hEPD_std.Reset()


In [11]:
#MAIN ANALYSIS CELL

#mult_bin = mult_bins_1[6]
mult_bin = high_mult_bin

num_jets_wta = 0 #number of jets that meet cuts
num_jets_std = 0 #should be same i think

num_pairs_wta = int(0) #counting number of signal pairs
num_pairs_std = int(0)

Nch_wta = 0 #total jet multiplicity for all jets in the bin
Nch_std = 0

#loopcount = 0 #for testing the loop
# this loop fills hSig and hEPD
for event in tree:
    #loopcount+=1 #for testing
    #if loopcount>40000:
    #    break

    # 1. Loop over jets in the event
    for ijet in range(event.genJetPt.size()):
        
        #extracting standard pt and eta from ROOT data for cuts
        std_pt = event.genJetPt[ijet]
        std_eta = event.genJetEta[ijet]
        
        if eventPass(std_pt, std_eta): #jet cut using STANDARD pt and eta

            # 2. Get the daughter vectors for ONLY this jet
            dau_pts = event.genDau_pt[ijet]
            dau_etas = event.genDau_eta[ijet]
            dau_phis = event.genDau_phi[ijet]
            dau_charges = event.genDau_chg[ijet]

            # 3. Create PseudoJets for this specific jet
            jet_constituents = make_pseudojets(dau_pts, dau_etas, dau_phis, dau_charges) #lab frame particle cuts are built in

            # 4. Cluster with WTA and standard
            wta_clustered = fastjet.ClusterSequence(jet_constituents, wta_def)  #this is a cluster sequence, can't be directly queried for momentum, etc.
            wta_jets_tup = fastjet.sorted_by_pt(wta_clustered.inclusive_jets()) #just one jet but it's a tuple so we need 0th index

            std_clustered = fastjet.ClusterSequence(jet_constituents, std_def)  #cluster sequence again
            std_jets_tup = fastjet.sorted_by_pt(std_clustered.inclusive_jets()) #tuple again
            
            if len(wta_jets_tup) > 0: #preventing indexing errors
                wta_jet = wta_jets_tup[0]   #indexing to get the actual pseudojet we can query
                std_jet = std_jets_tup[0]   
                
                # 5. shifting to jet frame
                #wta_in_jet_frame is a list of particle pseudojets in the jet frame (jet frame eta and jt cuts built in)
                #wta_mult is the jet multiplicity, which counts all particles in the clustered jet that pass lab-frame cuts
                wta_in_jet_frame, wta_mult = jetToJetFrame(wta_jet) 
                std_in_jet_frame, std_mult = jetToJetFrame(std_jet)
                
                #check mult bin
                if mult_bin_check(wta_mult, mult_bin): 
                    buildSignal(wta_in_jet_frame, hSig_wta, hEPD_wta) # fill in hSig and hEPD
                    num_pairs_wta += len(wta_in_jet_frame)*(len(wta_in_jet_frame) - 1) // 2 #(number of particles in jet) choose (2); double// for integer division
                    num_jets_wta += 1
                    Nch_wta += wta_mult 
    
                if mult_bin_check(std_mult, mult_bin):
                    buildSignal(std_in_jet_frame, hSig_std, hEPD_std)
                    num_pairs_std += len(std_in_jet_frame)*(len(std_in_jet_frame) - 1) // 2 
                    num_jets_std += 1
                    Nch_std += std_mult
                

#--------------------------------------------------------------------------
#                         FastJet release 3.5.1
#                 M. Cacciari, G.P. Salam and G. Soyez                  
#     A software package for jet finding and analysis at colliders      
#                           https://fastjet.fr                           
#	                                                                      
# Please cite EPJC72(2012)1896 [arXiv:1111.6097] if you use this package
# for scientific work and optionally PLB641(2006)57 [hep-ph/0512210].   
#                                                                       
# FastJet is provided without warranty under the GNU GPL v2 or higher.  
# It uses T. Chan's closest pair algorithm, S. Fortune's Voronoi code,
# CGAL and 3rd party plugin jet algorithms. See COPYING file for details.
#--------------------------------------------------------------------------


In [12]:
avg_Nch_wta = Nch_wta / num_jets_wta #finding average Nch per jet
avg_Nch_std = Nch_std / num_jets_std

Nassoc_wta = (num_pairs_wta / Nch_wta)/num_jets_wta  #average number of pairs per trigger particle in this bin (used in fourier fit)
Nassoc_std = (num_pairs_std / Nch_std)/num_jets_std

In [13]:
print("average Nch (wta) for this bin:", avg_Nch_wta)
print("average Nch (std) for this bin:", avg_Nch_std)

print("N assoc wta:", Nassoc_wta)
print("N assoc std:", Nassoc_std)

print("wta signal pairs:",num_pairs_wta)
print("std signal pairs:", num_pairs_std)

print("wta jets:", num_jets_wta)
print("std jets:", num_jets_std)

average Nch (wta) for this bin: 83.44444444444444
average Nch (std) for this bin: 86.07142857142857
N assoc wta: 1.7452285841100754
N assoc std: 1.3339063426200357
wta signal pairs: 11796
std signal pairs: 22503
wta jets: 9
std jets: 14


In [14]:
#fill background histograms
buildBkg(hEPD_wta, hBkg_wta, num_pairs_wta) #wta
buildBkg(hEPD_std, hBkg_std, num_pairs_std) #std


In [15]:
# Doing signal / background to get yield. 

#find B(0,0)
origin_bin_wta = hBkg_wta.FindBin(0.0, 0.0)     # 1. Find the global bin index that corresponds to x=0.0, y=0.0
B_00_wta = hBkg_wta.GetBinContent(origin_bin_wta)   # 2. Extract the number from that bin (this is B(0,0) , unscaled by ntrig)

origin_bin_std = hBkg_std.FindBin(0.0, 0.0)
B_00_std = hBkg_std.GetBinContent(origin_bin_std) 


#find yield
hYield_wta = hSig_wta.Clone("hYield_wta")
hYield_wta.Divide(hBkg_wta)      # This does S(dEta, dPhi) / B(dEta, dPhi)
hYield_wta.Scale(B_00_wta)       # This multiplies every bin by the single number B(0,0) (unscaled, but hBkg is also unscaled so it cancels)
hYield_wta.Scale(1/num_jets_wta)    #averaging over all jets

hYield_std = hSig_std.Clone("hYield_std")
hYield_std.Divide(hBkg_std) 
hYield_std.Scale(B_00_std)    
hYield_std.Scale(1/num_jets_std)  

In [46]:
# --- SAVING THE OUTPUT ---

# 1. Define your specific output folder path
output_dir = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/2pc_output_histograms"

# Create the folder if it doesn't already exist
os.makedirs(output_dir, exist_ok=True)

# 2. Create the dynamic filename
if len(mult_bin) == 2:
    bin_name = f"{mult_bin[0]}_{mult_bin[1]}"
else:
    bin_name = f"{mult_bin[0]}_up"

filename = f"Yield_Histograms_Mult_{bin_name}.root"

# 3. Combine the folder path and the filename
full_filepath = os.path.join(output_dir, filename)

# 4. Open the ROOT file using the FULL path
out_file = ROOT.TFile(full_filepath, "RECREATE")

# 5. Rename and write the histograms
hYield_wta.SetName(f"hYield_WTA_{bin_name}")
hYield_std.SetName(f"hYield_STD_{bin_name}")
hSig_wta.SetName(f"hSig_WTA_{bin_name}")
hSig_std.SetName(f"hSig_STD_{bin_name}")
hBkg_wta.SetName(f"hBkg_WTA_{bin_name}")
hBkg_std.SetName(f"hBkg_STD_{bin_name}")

hYield_wta.Write()
hYield_std.Write()
hSig_wta.Write()
hSig_std.Write()
hBkg_wta.Write()
hBkg_std.Write()

out_file.Close()

print(f"Successfully saved yields to {full_filepath}")

Successfully saved yields to /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/2pc_output_histograms/Yield_Histograms_Mult_80_up.root


# Now moving on to doing the 1d projection and v2 extraction

In [16]:
#projection for |dEta| > 2 
# (note:jet frame cuts excluded eta star > 5)

eta_min = 2 #from prl paper, abs(delta eta star) should be greater than 2
eta_max = 4 #all the delta eta values should be less than this anyways i think

def project(hYield):
    '''
    Makes the 1d projection of the yield histograms
    '''
    # 1. Bins within delta eta star range
    bin_low = hYield.GetXaxis().FindBin(eta_min)
    bin_high = hYield.GetXaxis().FindBin(eta_max)
    
    # 2. doing the projection onto delta phi star
    h1D = hYield.ProjectionY("h1D", bin_low, bin_high)

    # 3. normalising by delta phi star bin width
    phi_bw = h1D.GetBinWidth(1)
    h1D.Scale(1.0 / phi_bw)
    return h1D


In [18]:
#fourier fitting

#cosine series (param [0] should be N_assoc)
cosine_series = (
    "[0]/(2*TMath::Pi()) * (1 + 2*[1]*TMath::Cos(x) + 2*[2]*TMath::Cos(2*x) "
    "+ 2*[3]*TMath::Cos(3*x) + 2*[4]*TMath::Cos(4*x) + 2*[5]*TMath::Cos(5*x))"
)

# root fit function
fit_func = ROOT.TF1("fourier_fit", cosine_series, -0.5*math.pi , 1.5*math.pi)

def fourierFit(h1D, Nassoc):
    '''
    Applies a fourier fit to the 1d projection
    '''
    #seeding parameters
    fit_func.SetParameter(0, Nassoc)    #what it should be, based on the paper
    fit_func.SetParameter(1, 0.1)   # v1 seed
    fit_func.SetParameter(2, 0.1)  # v2 seed (elliptic flow)
    fit_func.SetParameter(3, 0.1)   # v3 seed
    fit_func.SetParameter(4, 0.1)   # v4 seed
    fit_func.SetParameter(5, 0.1)   # v5 seed       all the seeds are 0.1 in the DrawFlow.C file so I'm just using that


    # EXECUTE THE FIT
    # ==========================================
    # "R" forces the fit range specified in the TF1 definition
    # "M" tells ROOT to search for better minimums (improves v2 precision)
    # "E" invokes the advanced Minos error estimation
    # "Q" keeps the terminal output quiet
    h1D.Fit(fit_func, "R M E Q")

    #extract parameters
    v1 = fit_func.GetParameter(1)
    v2 = fit_func.GetParameter(2)  # This is your final Winner-Take-All elliptic flow value!
    v3 = fit_func.GetParameter(3)

    #get error
    v1_err = fit_func.GetParError(1)
    v2_err = fit_func.GetParError(2)
    v3_err = fit_func.GetParError(3)

    #this bit is in the github but idk if i need it:
    # Apply the sqrt(2) scaling factor found in DrawVn's source code to conservatively account for Signal/Background statistical correlation
    v2_err_final = v2_err * math.sqrt(2)

    coeffs = [v1, v2, v3]
    errs = [v1_err, v2_err, v3_err]
    return coeffs, errs


In [ ]:
#finding v2
h1D_wta = project(hYield_wta)
h1D_std = project(hYield_std)

([-0.2580674253602378, -0.05434290264109242, 0.11969974391269389], [0.029470610703217, 0.03415101202670449, 0.040116538540910796])
([-0.2580644189853301, -0.054343353615167955, 0.11969764431445784], [0.029470984889634705, 0.034151263156176545, 0.04011690314851653])


Info in <TCanvas::MakeDefCanvas>:  created default TCanvas with name c1


In [20]:
fit_wta = fourierFit(h1D_wta, Nassoc_wta)
fit_std = fourierFit(h1D_std, Nassoc_std)

wta_v2 = [fit_wta[0][1], fit_wta[1][1]] #v2, error
std_v2 = [fit_std[0][1], fit_std[1][1]] #v2, error

print(fit_wta)
print(fit_std)
print(wta_v2)
print(std_v2)

([-0.25806749047561195, -0.05434285563133452, 0.11969976340287987], [0.029470604820453012, 0.03415101160610796, 0.040116534441636174])
([-0.25806440225075605, -0.05434350600973992, 0.11969781980682082], [0.029470980617549023, 0.03415125069234023, 0.040116882831947774])
[-0.05434285563133452, 0.03415101160610796]
[-0.05434350600973992, 0.03415125069234023]
